# 🚀 Notebook — API REST con FastAPI
### Sistema Ernesto Investing AI — Apoyo en Decisiones de Inversión con IA

**Curso:** Introducción al Desarrollo de Software (iDeSo) — UNMSM · FISI  
**Prof.:** Mg. Ing. Ernesto D. Cancho-Rodríguez, MBA (GWU)  
**Semana 11 — Fase 2: API REST con FastAPI**

---

## 🎯 Objetivo

Levantar la API REST del sistema usando **FastAPI + uvicorn**, exponerla a Internet mediante
un túnel **ngrok** y verificar todos los endpoints con **Swagger UI**.

## 📋 Endpoints expuestos

| Método | Endpoint | Descripción |
|--------|----------|-------------|
| GET | `/api/salud` | Estado de salud del servidor |
| GET | `/api/mercado/{ticker}` | Datos OHLCV + indicadores técnicos |
| GET | `/api/svc/{ticker}` | Señales y métricas del clasificador SVC |
| GET | `/api/rnns/{ticker}` | Señales y métricas LSTM/BiLSTM/GRU/SimpleRNN |
| GET | `/api/lstm/{ticker}?horizonte=30` | Predicciones del regresor LSTM |
| GET | `/api/nlp` | Análisis de sentimiento NLP (VADER) |

## 🪙 Tickers disponibles
`FSM` · `VOLCABC1.LM` · `ABX.TO` · `BVN` · `BHP`

> ⚠️ Los endpoints `/api/svc/` y `/api/rnns/` entrenan modelos en tiempo real.
> Se recomienda habilitar GPU en Colab: **Entorno de ejecución → Cambiar tipo de entorno → GPU**

---
## Celda 1 — Instalación de dependencias

In [ ]:
# Instalación de todas las librerías necesarias para la API
# Ejecutar esta celda una única vez al inicio de la sesión
!pip install fastapi uvicorn pyngrok nest-asyncio \
             yfinance ta scikit-learn \
             tensorflow nltk \
             numpy pandas --quiet

print('✅ Dependencias instaladas correctamente')

  Preparing metadata (setup.py) ... done
✅ Dependencias instaladas correctamente


---
## Celda 2 — Subir el archivo main.py a Colab

Ejecutar la celda de abajo para subir `main.py` desde tu equipo local a este entorno de Colab.
Alternativamente, si ya tienes `main.py` en un repositorio de GitHub, puedes clonarlo con:
```bash
!git clone https://github.com/TU_USUARIO/TU_REPO.git
%cd TU_REPO
```

In [ ]:
# Opción A — Subir main.py manualmente desde tu computadora
from google.colab import files
print('📂 Selecciona el archivo main.py desde tu computadora...')
uploaded = files.upload()

if 'main.py' in uploaded:
    print('✅ main.py subido correctamente al entorno de Colab')
else:
    print('⚠️  No se detectó main.py. Asegúrate de seleccionar el archivo correcto.')

📂 Selecciona el archivo main.py desde tu computadora...


Saving main.py to main.py
✅ main.py subido correctamente al entorno de Colab


In [ ]:
# Verificar que main.py existe en el entorno
import os
if os.path.exists('main.py'):
    tam = os.path.getsize('main.py') / 1024
    print(f'✅ main.py encontrado — Tamaño: {tam:.1f} KB')
    # Mostrar los primeros 30 caracteres del archivo
    with open('main.py', 'r') as f:
        lineas = f.readlines()[:5]
    print('   Primeras líneas:')
    for l in lineas:
        print('  ', l.rstrip())
else:
    print('❌ main.py NO encontrado. Ejecutar la celda anterior primero.')

✅ main.py encontrado — Tamaño: 43.2 KB
   Primeras líneas:
   # =============================================================================
   # main.py — API REST con FastAPI
   # Sistema Ernesto Investing AI — Apoyo en Decisiones de Inversión con IA
   # Introducción al Desarrollo de Software (iDeSo) — UNMSM · FISI
   # Prof. Mg. Ing. Ernesto D. Cancho-Rodríguez, MBA (GWU)


---
## Celda 3 — Configurar ngrok

**ngrok** crea un túnel seguro que expone el servidor local de Colab a Internet.

1. Ir a [https://dashboard.ngrok.com/signup](https://dashboard.ngrok.com/signup) y crear una cuenta gratuita.
2. Copiar el **Authtoken** desde [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
3. Pegarlo en la variable `NGROK_TOKEN` de la celda de abajo.

In [ ]:
from pyngrok import ngrok, conf

# ── REEMPLAZAR con tu token de ngrok ─────────────────────────────────
NGROK_TOKEN = '3FgSrbkZPkGAOS9J1liS74h5PWw_26upiRfYgp3AzKyJAjw2C'
# ─────────────────────────────────────────────────────────────────────

ngrok.set_auth_token(NGROK_TOKEN)
print('✅ Token de ngrok configurado correctamente')

✅ Token de ngrok configurado correctamente


---
## Celda 4 — Lanzar la API FastAPI con ngrok

> ⚡ Esta celda **bloquea** el kernel mientras el servidor esté activo (comportamiento esperado).  
> Para detener el servidor, ir a **Entorno de ejecución → Interrumpir ejecución**.

In [ ]:
import threading
import uvicorn
import nest_asyncio
import socket
from pyngrok import ngrok, conf

nest_asyncio.apply()

# ── Cerrar túneles ngrok anteriores ──────────────────────────────────
ngrok.kill()

# ── Verificar si el puerto 8000 está en uso y liberarlo ──────────────
def puerto_libre(puerto):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', puerto)) != 0

if not puerto_libre(8000):
    import os, signal, subprocess
    result = subprocess.run(['fuser', '-k', '8000/tcp'],
                            capture_output=True, text=True)
    print('⚠️  Puerto 8000 liberado — reintentando...')
    import time; time.sleep(2)

# ── Importar la app FastAPI ───────────────────────────────────────────
from main import app

# ── Iniciar uvicorn en hilo separado ─────────────────────────────────
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run, daemon=True)
thread.start()

import time; time.sleep(2)  # dar tiempo a que arranque

# ── Abrir túnel ngrok ─────────────────────────────────────────────────
tunnel = ngrok.connect(8000)
URL_PUBLICA = tunnel.public_url

print('=' * 66)
print('  🚀 Ernesto Investing AI — API REST ACTIVA')
print('=' * 66)
print(f'  URL pública (ngrok) : {URL_PUBLICA}')
print(f'  Swagger UI          : {URL_PUBLICA}/docs')
print(f'  Health Check        : {URL_PUBLICA}/api/salud')
print('=' * 66)
print(f'    {URL_PUBLICA}/api/mercado/BVN')
print(f'    {URL_PUBLICA}/api/svc/FSM')
print(f'    {URL_PUBLICA}/api/rnns/BHP')
print(f'    {URL_PUBLICA}/api/lstm/ABX.TO?horizonte=30')
print(f'    {URL_PUBLICA}/api/nlp')
print('=' * 66)

  🚀 Ernesto Investing AI — API REST ACTIVA
  URL pública (ngrok) : https://devotee-dinghy-slapstick.ngrok-free.dev
  Swagger UI          : https://devotee-dinghy-slapstick.ngrok-free.dev/docs
  Health Check        : https://devotee-dinghy-slapstick.ngrok-free.dev/api/salud
    https://devotee-dinghy-slapstick.ngrok-free.dev/api/mercado/BVN
    https://devotee-dinghy-slapstick.ngrok-free.dev/api/svc/FSM
    https://devotee-dinghy-slapstick.ngrok-free.dev/api/rnns/BHP
    https://devotee-dinghy-slapstick.ngrok-free.dev/api/lstm/ABX.TO?horizonte=30
    https://devotee-dinghy-slapstick.ngrok-free.dev/api/nlp


---
## Celda 5 — Verificación de endpoints (ejecutar en otra sesión o después de lanzar)

Puedes probar los endpoints directamente desde Colab usando la librería `requests`.

> ⚠️ Para ejecutar esta celda mientras el servidor está activo, necesitas abrir un **segundo notebook**
> o probar desde el navegador vía Swagger UI (`{URL_PUBLICA}/docs`).

In [ ]:
import requests

# ── Pegar aquí la URL que imprimió la Celda 4 ────────────────────────
URL_BASE = 'https://devotee-dinghy-slapstick.ngrok-free.dev'  # tu URL real

headers = {'ngrok-skip-browser-warning': 'true'}  # evita pantalla de aviso

endpoints = [
    '/api/salud',
    '/api/mercado/BVN',
    '/api/nlp',
]

print('🔍 Verificando endpoints...\n')
for ep in endpoints:
    try:
        r = requests.get(URL_BASE + ep, headers=headers, timeout=30)
        claves = list(r.json().keys()) if r.status_code == 200 else '—'
        estado = '✅' if r.status_code == 200 else '❌'
        print(f'  {estado} {ep}  →  {r.status_code}  |  claves: {claves}')
    except Exception as e:
        print(f'  ⚠️  {ep}  →  {str(e)[:80]}')

🔍 Verificando endpoints...

  ✅ /api/salud  →  200  |  claves: ['estado', 'sistema', 'version', 'timestamp', 'tickers_disponibles', 'periodo_datos', 'modelos', 'framework']
  ✅ /api/mercado/BVN  →  200  |  claves: ['metadata', 'fechas', 'ohlcv', 'indicadores', 'senales', 'ultimo']
  ✅ /api/nlp  →  200  |  claves: ['metadata', 'noticias', 'resumen_tickers', 'mercado']


---
## 📌 Notas para el Entregable

1. **Swagger UI** (`/docs`) debe estar accesible y mostrar los 6 endpoints documentados.
2. Incluir **capturas de pantalla** de Swagger UI con al menos 2 endpoints ejecutados exitosamente.
3. Copiar la **URL pública de ngrok** en el informe como evidencia del despliegue.
4. El endpoint `/api/salud` sirve como **health check** para verificar el estado del servidor.

### Conexión con el Frontend (Fase 3)

Para conectar los archivos HTML del frontend a esta API, reemplazar las llamadas a
`generarDatos()` / `Math.random()` por un `fetch()` a la URL pública de ngrok:

```javascript
// Ejemplo: ernesto_investing_dashboard_mercado.html
const URL_API = 'https://TU_URL_NGROK.ngrok.io';

fetch(`${URL_API}/api/mercado/BVN`)
  .then(r => r.json())
  .then(datos => {
    // Usar los datos reales para renderizar el dashboard
    renderizarGrafico(datos);
  });
```

### Estructura de archivos entregables

```
📁 Semana11_Backend_iDeSo/
├── 📄 main.py                        ← Entregable 4: API REST con FastAPI
├── 📓 notebook5_fastapi_ngrok.ipynb  ← Este notebook
├── 📓 notebook2_svc_clasificador.ipynb
├── 📓 Notebook3_Clasificadores_RNN_iDeSo.ipynb
├── 📓 notebook4_regresor_lstm.ipynb
└── 📁 json/
    ├── datos_svc.json
    ├── datos_rnns_clf.json
    ├── lstm_regressor_data.json
    └── datos_nlp.json
```